In [ ]:
import torch
import pickle
from vmc_torch.experiment.vmap.vmap_torch_utils import RobustSVD
import autoray as ar

import quimb.tensor as qtn
from vmc_torch.experiment.vmap.GPU.GPU_vmap_utils import sample_next, evaluate_energy, compute_grads, random_initial_config
from vmc_torch.experiment.vmap.vmap_models import PEPS_Model, fPEPS_Model


JITTER = 0
driver='gesvd'
ar.register_function('torch', 'linalg.svd', lambda x, **kwargs: RobustSVD.apply(x, jitter_strength=JITTER, driver=driver))


# 设置默认精度
torch.set_default_dtype(torch.float64)
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')
torch.set_default_device(device)

# ==========================================
# 2. 参数设置与模型加载
# ==========================================
Lx, Ly = 4, 4
nsites = Lx * Ly
N_f = nsites - 2
D = 4
chi = 16

# 路径配置 (保持你的原样)
pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/data'
u1z2 = True
appendix = '_U1SU' if u1z2 else ''

# 加载骨架 (这部分很快，可以在 CPU 做完再转 GPU)
# 注意：pickle load 最好只在 Rank 0 做然后广播，或者大家各自读文件(如果文件系统支持并发)
# 这里假设大家各自读文件没问题
params_pkl = pickle.load(open(pwd+f'/{Lx}x{Ly}/t=1.0_U=8.0/N={N_f}/Z2/D={D}/peps_su_params{appendix}.pkl', 'rb'))
skeleton = pickle.load(open(pwd+f'/{Lx}x{Ly}/t=1.0_U=8.0/N={N_f}/Z2/D={D}/peps_skeleton{appendix}.pkl', 'rb'))
peps = qtn.unpack(params_pkl, skeleton)

# 预处理 (CPU)
for ts in peps.tensors:
    ts.modify(data=ts.data.to_flat() * 4)
for site in peps.sites:
    peps[site].data._label = site
    peps[site].data.indices[-1]._linearmap = ((0, 0), (1, 0), (1, 1), (0, 1))

# 初始化模型并移动到 GPU
# fpeps_model = Transformer_fPEPS_Model_batchedAttn(
#     tn=peps, max_bond=chi, embed_dim=8, attn_heads=4, nn_hidden_dim=16, nn_eta=1, dtype=torch.float64,
# )
fpeps_model = fPEPS_Model(
    tn=peps, max_bond=chi, dtype=torch.float64,
)
fpeps_model.to(device) # <--- 关键：模型全在 GPU
# 尝试编译模型
c_fpeps_model = torch.compile(fpeps_model, mode="default", fullgraph=False)

In [3]:
n_params = sum(p.numel() for p in fpeps_model.parameters())
# ==========================================
# 3. 采样配置
# ==========================================

# 并行运行的 Chain 数量 (Batch Size)
# 如果显存够，可以直接设为 samples_per_rank，这样一步到位
# 如果显存不够，可以设小一点，循环多次累积
batch_size_per_rank = 64
# 确保初始化 walkers 在 GPU 上
fxs_list = [random_initial_config(N_f, nsites, seed=42+_) for _ in range(batch_size_per_rank)]
fxs = torch.stack(fxs_list).to(device)

In [4]:
fpeps_model(fxs)

tensor([-2.8608e+01, -4.7005e+00,  3.2531e-01,  1.1548e+02,  1.9528e+03,
         5.2308e-01,  7.0860e-01,  1.1743e+00,  1.0720e+02, -7.0518e-01,
         3.2292e+00,  2.1395e+00,  9.7898e+01, -1.9431e+02,  1.1213e+02,
        -1.1331e+01,  6.2423e+00, -1.9171e+02,  4.8512e+00, -1.9116e+01,
         8.4375e+00,  1.0579e+01, -3.3270e+01, -9.9083e+00,  7.2361e-02,
        -2.4135e+03,  6.0247e+00,  6.9989e-01,  7.3308e-01, -7.2558e+02,
        -4.2854e+01, -1.7414e+00, -6.3953e+00, -2.1817e+00,  6.1778e+01,
         2.1092e+01,  5.1735e+01, -1.2347e+00, -1.4300e+01, -2.4318e+01,
        -9.8329e+02,  1.5655e+01,  8.0222e-04, -2.4979e+00,  3.1930e+00,
        -2.2966e-01,  7.6330e+00, -5.6677e-01, -3.6860e+02, -1.7214e+01,
        -6.9745e-01,  5.1162e-01,  2.6744e+00, -1.3242e+03,  3.1987e+01,
         1.7054e+00, -2.2305e+01, -1.8246e-01,  6.4935e+02, -1.1587e+03,
        -9.1717e+01, -1.8021e+01, -1.0570e+03, -8.6327e+02],
       grad_fn=<MulBackward0>)

In [6]:
c_fpeps_model(fxs)

tensor([-2.8608e+01, -4.7005e+00,  3.2531e-01,  1.1548e+02,  1.9528e+03,
         5.2308e-01,  7.0860e-01,  1.1743e+00,  1.0720e+02, -7.0518e-01,
         3.2292e+00,  2.1395e+00,  9.7898e+01, -1.9431e+02,  1.1213e+02,
        -1.1331e+01,  6.2423e+00, -1.9171e+02,  4.8512e+00, -1.9116e+01,
         8.4375e+00,  1.0579e+01, -3.3270e+01, -9.9083e+00,  7.2361e-02,
        -2.4135e+03,  6.0247e+00,  6.9989e-01,  7.3308e-01, -7.2558e+02,
        -4.2854e+01, -1.7414e+00, -6.3953e+00, -2.1817e+00,  6.1778e+01,
         2.1092e+01,  5.1735e+01, -1.2347e+00, -1.4300e+01, -2.4318e+01,
        -9.8329e+02,  1.5655e+01,  8.0222e-04, -2.4979e+00,  3.1930e+00,
        -2.2966e-01,  7.6330e+00, -5.6677e-01, -3.6860e+02, -1.7214e+01,
        -6.9745e-01,  5.1162e-01,  2.6744e+00, -1.3242e+03,  3.1987e+01,
         1.7054e+00, -2.2305e+01, -1.8246e-01,  6.4935e+02, -1.1587e+03,
        -9.1717e+01, -1.8021e+01, -1.0570e+03, -8.6327e+02],
       grad_fn=<MulBackward0>)

In [10]:
%%time
fpeps_model(fxs)

CPU times: user 960 ms, sys: 3.32 ms, total: 963 ms
Wall time: 114 ms


tensor([-2.8608e+01, -4.7005e+00,  3.2531e-01,  1.1548e+02,  1.9528e+03,
         5.2308e-01,  7.0860e-01,  1.1743e+00,  1.0720e+02, -7.0518e-01,
         3.2292e+00,  2.1395e+00,  9.7898e+01, -1.9431e+02,  1.1213e+02,
        -1.1331e+01,  6.2423e+00, -1.9171e+02,  4.8512e+00, -1.9116e+01,
         8.4375e+00,  1.0579e+01, -3.3270e+01, -9.9083e+00,  7.2361e-02,
        -2.4135e+03,  6.0247e+00,  6.9989e-01,  7.3308e-01, -7.2558e+02,
        -4.2854e+01, -1.7414e+00, -6.3953e+00, -2.1817e+00,  6.1778e+01,
         2.1092e+01,  5.1735e+01, -1.2347e+00, -1.4300e+01, -2.4318e+01,
        -9.8329e+02,  1.5655e+01,  8.0222e-04, -2.4979e+00,  3.1930e+00,
        -2.2966e-01,  7.6330e+00, -5.6677e-01, -3.6860e+02, -1.7214e+01,
        -6.9745e-01,  5.1162e-01,  2.6744e+00, -1.3242e+03,  3.1987e+01,
         1.7054e+00, -2.2305e+01, -1.8246e-01,  6.4935e+02, -1.1587e+03,
        -9.1717e+01, -1.8021e+01, -1.0570e+03, -8.6327e+02],
       grad_fn=<MulBackward0>)

In [9]:
%%time
c_fpeps_model(fxs)

CPU times: user 1.14 s, sys: 26.9 ms, total: 1.17 s
Wall time: 364 ms


tensor([-2.8608e+01, -4.7005e+00,  3.2531e-01,  1.1548e+02,  1.9528e+03,
         5.2308e-01,  7.0860e-01,  1.1743e+00,  1.0720e+02, -7.0518e-01,
         3.2292e+00,  2.1395e+00,  9.7898e+01, -1.9431e+02,  1.1213e+02,
        -1.1331e+01,  6.2423e+00, -1.9171e+02,  4.8512e+00, -1.9116e+01,
         8.4375e+00,  1.0579e+01, -3.3270e+01, -9.9083e+00,  7.2361e-02,
        -2.4135e+03,  6.0247e+00,  6.9989e-01,  7.3308e-01, -7.2558e+02,
        -4.2854e+01, -1.7414e+00, -6.3953e+00, -2.1817e+00,  6.1778e+01,
         2.1092e+01,  5.1735e+01, -1.2347e+00, -1.4300e+01, -2.4318e+01,
        -9.8329e+02,  1.5655e+01,  8.0222e-04, -2.4979e+00,  3.1930e+00,
        -2.2966e-01,  7.6330e+00, -5.6677e-01, -3.6860e+02, -1.7214e+01,
        -6.9745e-01,  5.1162e-01,  2.6744e+00, -1.3242e+03,  3.1987e+01,
         1.7054e+00, -2.2305e+01, -1.8246e-01,  6.4935e+02, -1.1587e+03,
        -9.1717e+01, -1.8021e+01, -1.0570e+03, -8.6327e+02],
       grad_fn=<MulBackward0>)